In [ ]:
!pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install --upgrade transformers
!pip install librosa scikit-learn tqdm matplotlib seaborn

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import librosa
from transformers import WhisperFeatureExtractor
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
# Kaggle dataset paths
TESS_DIR = "/kaggle/input/datasets/orvile/toronto-emotional-speech-set-tess"
RAVDESS_DIR = "/kaggle/input/datasets/uwrfkaggler/ravdess-emotional-speech-audio"

In [ ]:
import os
from collections import Counter
import pandas as pd

EMOTION_MAP = {
    '01': 0, '02': 0, '03': 1, '04': 2, '05': 3, '06': 4, '07': 5, '08': 6,
    'neutral': 0, 'happy': 1, 'sad': 2, 'angry': 3, 'fear': 4, 'disgust': 5, 'ps': 6
}
LABEL_NAMES = ['neutral', 'happy', 'sad', 'angry', 'fear', 'disgust', 'surprise']

filepaths = []
labels = []

# RAVDESS
for root, dirs, files in os.walk(RAVDESS_DIR):
    for f in files:
        if f.endswith(".wav"):
            parts = f.replace('.wav','').split('-')
            if len(parts) >= 3:
                emotion_code = parts[2]
                if emotion_code in EMOTION_MAP:
                    filepaths.append(os.path.join(root,f))
                    labels.append(EMOTION_MAP[emotion_code])

# TESS
for root, dirs, files in os.walk(TESS_DIR):
    folder = os.path.basename(root).lower()
    for f in files:
        if f.endswith(".wav"):
            for key in EMOTION_MAP:
                if folder.endswith(key):
                    filepaths.append(os.path.join(root,f))
                    labels.append(EMOTION_MAP[key])
                    break

# Convert to DataFrame
df = pd.DataFrame({"filepath": filepaths, "label": labels})
df.head()

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df, test_size=0.2, stratify=df['label'], random_state=42
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42
)

In [ ]:
test_dataset = AudioDataset(test_df, feature_extractor)

from torch.utils.data import DataLoader

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False
)

In [ ]:
import torch
from torch.utils.data import Dataset
import librosa
from transformers import WhisperFeatureExtractor

feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-base")

class AudioDataset(Dataset):
    def __init__(self, dataframe, feature_extractor, max_length=30):
        self.df = dataframe
        self.feature_extractor = feature_extractor
        self.max_length = max_length  # seconds

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio, sr = librosa.load(row['filepath'], sr=16000)
        # Pad or trim
        if len(audio) > sr * self.max_length:
            audio = audio[:sr*self.max_length]
        else:
            audio = np.pad(audio, (0, sr*self.max_length - len(audio)))
        input_features = self.feature_extractor(audio, sampling_rate=sr, return_tensors="pt").input_features
        label = torch.tensor(row['label'], dtype=torch.long)
        return {"input_features": input_features.squeeze(0), "labels": label}

In [ ]:
from torch.utils.data import DataLoader

train_dataset = AudioDataset(train_df, feature_extractor)
val_dataset = AudioDataset(val_df, feature_extractor)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

In [ ]:
import torch
import torch.nn as nn
from transformers import WhisperModel

class WhisperSERModel(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.whisper = WhisperModel.from_pretrained(model_name).encoder
        self.whisper.gradient_checkpointing_enable()  # save memory
        hidden_size = self.whisper.config.d_model  # 768 for whisper-base

        # Attention pooling
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.Tanh(),
            nn.Linear(256, 1),
            nn.Softmax(dim=1)
        )

        # Classifier head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_labels)
        )

    def forward(self, input_features, labels=None):
        # input_features: [batch, seq_len, feature_dim]
        hidden_states = self.whisper(input_features).last_hidden_state  # [batch, seq, hidden]
        weights = self.attention(hidden_states)  # [batch, seq, 1]
        pooled = torch.sum(weights * hidden_states, dim=1)  # [batch, hidden]
        logits = self.classifier(pooled)

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)
        return {"loss": loss, "logits": logits}

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

CONFIG = {
    "model_name": "openai/whisper-base",
    "num_labels": 7,
    "epochs": 5,
    "lr": 3e-5,
    "batch_size": 8,
    "gradient_accumulation_steps": 2,
    "output_dir": "./model_output"
}

import os
os.makedirs(CONFIG["output_dir"], exist_ok=True)

model = WhisperSERModel(CONFIG["model_name"], CONFIG["num_labels"]).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"])
total_steps = (len(train_loader) // CONFIG["gradient_accumulation_steps"]) * CONFIG["epochs"]

from transformers import get_linear_schedule_with_warmup
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=0, num_training_steps=total_steps
)

In [ ]:
def set_freeze(model, freeze=True):
    for param in model.whisper.parameters():
        param.requires_grad = not freeze

In [ ]:
set_freeze(model, freeze=True)

In [ ]:
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score

train_losses, val_accuracies, val_f1s = [], [], []
best_f1 = 0
best_epoch = 0

for epoch in range(1, CONFIG["epochs"] + 1):
    # Unfreeze after 2 epochs
    if epoch == 3:
        set_freeze(model, freeze=False)
        print("Unfreezing Whisper encoder for fine-tuning.")

    model.train()
    running_loss = 0

    for i, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch}")):
        input_features = batch['input_features'].to(device)
        labels = batch['labels'].to(device)
        out = model(input_features, labels)
        loss = out['loss'] / CONFIG['gradient_accumulation_steps']
        loss.backward()
        running_loss += out['loss'].item()

        if (i + 1) % CONFIG['gradient_accumulation_steps'] == 0:
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

    # Validation
    model.eval()
    preds, actuals = [], []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation"):
            input_features = batch['input_features'].to(device)
            labels = batch['labels'].to(device)
            out = model(input_features)
            preds.extend(out['logits'].argmax(-1).cpu().numpy())
            actuals.extend(labels.cpu().numpy())

    acc = accuracy_score(actuals, preds)
    f1 = f1_score(actuals, preds, average='macro')
    train_losses.append(running_loss / len(train_loader))
    val_accuracies.append(acc)
    val_f1s.append(f1)

    if f1 > best_f1:
        best_f1 = f1
        best_epoch = epoch
        torch.save(model.state_dict(), os.path.join(CONFIG['output_dir'], "best_model.pt"))

    print(f"Epoch {epoch} | Loss: {running_loss/len(train_loader):.4f} | Acc: {acc:.4f} | F1: {f1:.4f}")

print(f"\nBest F1: {best_f1:.4f} at epoch {best_epoch}")

In [ ]:
model.load_state_dict(torch.load("./model_output/best_model.pt"))
model.eval()

preds, actuals = [], []

with torch.no_grad():
    for batch in test_loader:
        input_features = batch['input_features'].to(device)
        labels = batch['labels'].to(device)

        out = model(input_features)
        preds.extend(out['logits'].argmax(-1).cpu().numpy())
        actuals.extend(labels.cpu().numpy())

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(actuals, preds, target_names=LABEL_NAMES))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(actuals, preds)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=LABEL_NAMES,
            yticklabels=LABEL_NAMES)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
import json

summary = {
    "model": "whisper-base SER",
    "accuracy": accuracy_score(actuals, preds),
    "f1_macro": f1_score(actuals, preds, average='macro'),
    "dataset": "RAVDESS + TESS",
}

with open("results.json", "w") as f:
    json.dump(summary, f, indent=2)